# Part 1 — Project & AOI setup

**Output:** `aoi` (table asset). **DoD:** AOI renders over Goiás+DF; `config/datasets.yaml` pinned and verified against the live catalog.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import ee, geemap
import utils, features
project = utils.init()
print('EE initialized; project =', project)

### Build the AOI (GAUL level-1: Goiás + Distrito Federal)

In [ ]:
aoi_fc = features.aoi_fc()
print('states:', aoi_fc.aggregate_array('ADM1_NAME').getInfo())
aoi = features.aoi_geometry()
print('AOI area (km^2):', round(aoi.area(1000).divide(1e6).getInfo(), 1))

In [ ]:
Map = geemap.Map()
Map.centerObject(aoi, 7)
Map.addLayer(aoi_fc, {'color': 'red'}, 'AOI (GO+DF)')
Map

### Verify-at-build — confirm every pinned dataset ID / band exists
Print actual band names so `datasets.yaml` (esp. `VERIFY`-tagged soil scales) can be corrected before downstream parts run.

In [ ]:
ds = utils.cfg()
checks = {
    'TerraClimate': ee.ImageCollection(ds['climate']['terraclimate']['id']).first(),
    'SRTM': ee.Image(ds['terrain']['srtm']),
    'HydroSHEDS_ACC': ee.Image(ds['terrain']['hydrosheds_acc']),
    'GSW': ee.Image(ds['water']['gsw']),
    'MODIS': ee.ImageCollection(ds['phenology']['modis']).first(),
    'Oxford': ee.Image(ds['access']['oxford']),
    'WorldCover': ee.ImageCollection(ds['landcover']['worldcover']).first(),
}
for k, v in checks.items():
    print(f'{k:16s}', v.bandNames().getInfo())
for key, spec in ds['soil']['layers'].items():
    print(f'soil.{key:5s}', ee.Image(spec['id']).bandNames().getInfo())

### Export the AOI as an EE asset

In [ ]:
utils.ensure_folder(project)
task = utils.export_table(features.aoi_dissolved_fc(), project, 'aoi')
print('export', task.status()['description'], '->', task.status()['state'])